# Realized-Volatility Aftershock NIFTY Strangle

**A mechanical, regime-conditioned short-strangle backtest on NIFTY 50
weekly options**

This notebook is self-contained. It inventories the supplied expiry ZIPs,
rebuilds the spot series, constructs lagged signals, selects option
contracts, runs the minute-level backtest, exports a trade log, and
performs robustness checks.

The central hypothesis is **realized-volatility aftershock mean
reversion**: after a volatility shock, a 20-session realized-volatility
estimate can remain elevated even after the option market's current
priced move has normalized. When the ATM straddle's priced move is no
greater than the lagged expected absolute realized move, a systematically
sold, wider-break-even strangle may benefit from subsequent normalization
and time decay.

## Mechanical specification

- **Instrument:** NIFTY short strangle using the immediately following
  listed expiry.
- **Signal time:** 15:19 on the preceding NIFTY expiry; execution begins
  at 15:20, so the signal always precedes the fill.
- **ATM priced move:** contemporaneous ATM call close plus ATM put close.
- **Realized comparator:** 20-session sample standard deviation of lagged
  daily log returns, scaled by `sqrt(actual holding sessions) × sqrt(2/pi)`.
- **Entry regime:** priced-move percentage divided by the realized
  comparator must be no greater than 1.0.
- **Strikes:** the first quoted paired put at or below one priced move
  under spot and call at or above one priced move over spot.
- **Exits:** 50% premium decay, a 2× premium stop, or 15:20 on expiry.
  Minute-close triggers execute at the next common minute open, within
  five minutes.
- **Friction:** adverse 0.5% all-in adjustment on every option transaction.
- **Sizing:** integer lots constrained by a 1% stop-risk budget and 25%
  capital allocation under a stated 10% notional margin proxy.
- **Data boundary:** observations after 30 November 2025 are rejected in
  code.

In [1]:
import csv
import gzip
import io
import json
import math
import os
import statistics
import zipfile
from collections import Counter
from dataclasses import asdict, replace
from datetime import date, datetime, time
from pathlib import Path

OUTPUT_ROOT = Path("deliverables/notebook_outputs")
OUTPUT_ROOT.mkdir(exist_ok=True)

configured = os.environ.get("NIFTY_DATA_ROOT")
candidates = ([Path(configured)] if configured else []) + [
    Path("data/raw/nifty"),
]
DATA_ROOT = next((path for path in candidates if path.is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Set NIFTY_DATA_ROOT or place the supplied ZIPs in data/raw/nifty"
    )
print("Raw data folder configured successfully")

Raw data folder configured successfully


## Reusable strategy engine

The following cell contains the complete reader, feature calculations,
historical lot-size rules, contract selection, execution logic, sizing,
and P&L implementation. No hidden package or proprietary backtester is
used.

In [2]:
"""Reproducible NIFTY weekly short-strangle research engine.

Raw option CSV files are streamed directly from the provided expiry ZIPs.  The
module intentionally uses only Python's standard library so the research logic
does not depend on a private runtime.
"""

from __future__ import annotations

import csv
import gzip
import io
import math
import re
import statistics
import zipfile
from dataclasses import asdict, dataclass
from datetime import date, datetime, time, timedelta
from pathlib import Path
from typing import Iterable


OPTION_MEMBER_RE = re.compile(
    r"^(?P<strike>\d+)(?P<right>CE|PE)_(?P<expiry>\d{8})\.csv$"
)
TIMESTAMP_FORMAT = "%d-%m-%Y %H:%M:%S"
DATA_CUTOFF = date(2025, 11, 30)


@dataclass(frozen=True)
class Bar:
    timestamp: datetime
    open: float
    high: float
    low: float
    close: float
    volume: int
    oi: int
    ticker: str


@dataclass(frozen=True)
class StrategyConfig:
    signal_time: time = time(15, 19)
    planned_fill_time: time = time(15, 20)
    max_fill_delay_minutes: int = 5
    fallback_holding_sessions: int = 5
    realized_lookback: int = 20
    minimum_vrp_ratio: float = 1.10
    maximum_vrp_ratio: float = 1.00
    vrp_filter_direction: str = "above"
    use_trend_filter: bool = True
    maximum_trend_z: float = 1.00
    strike_width_multiplier: float = 1.00
    profit_target_fraction: float = 0.50
    stop_multiple: float = 2.00
    execution_friction: float = 0.005
    initial_capital: float = 2_000_000.0
    risk_fraction: float = 0.01
    margin_allocation_fraction: float = 0.25
    margin_proxy_fraction: float = 0.10
    use_regime_filter: bool = True

    def validate(self) -> None:
        if self.fallback_holding_sessions <= 0 or self.realized_lookback < 2:
            raise ValueError("Holding period and lookback must be positive")
        if not 0 < self.profit_target_fraction < 1:
            raise ValueError("Profit target must be between zero and one")
        if self.stop_multiple <= 1:
            raise ValueError("Stop multiple must exceed one")
        if not 0 <= self.execution_friction < 1:
            raise ValueError("Execution friction must be in [0, 1)")
        if self.strike_width_multiplier <= 0:
            raise ValueError("Strike width multiplier must be positive")
        if self.vrp_filter_direction not in {"above", "below"}:
            raise ValueError("VRP filter direction must be 'above' or 'below'")


@dataclass(frozen=True)
class RegimeFeatures:
    daily_sigma: float
    expected_abs_realized_move_pct: float
    implied_move_pct: float
    vrp_ratio: float
    five_day_trend_log_return: float
    trend_z: float


@dataclass
class Trade:
    mode: str
    expiry: str
    archive: str
    entry_date: str
    holding_sessions: int
    feature_end_date: str
    signal_timestamp: str
    entry_timestamp: str
    trigger_timestamp: str
    exit_timestamp: str
    exit_reason: str
    spot_at_signal: float
    atm_strike: int
    atm_call_close: float
    atm_put_close: float
    implied_move_points: float
    implied_move_pct: float
    expected_abs_realized_move_pct: float
    vrp_ratio: float
    trend_z: float
    put_strike: int
    call_strike: int
    put_entry_market: float
    call_entry_market: float
    gross_entry_mark: float
    put_entry_fill: float
    call_entry_fill: float
    net_entry_credit_points: float
    put_exit_market: float
    call_exit_market: float
    put_exit_fill: float
    call_exit_fill: float
    net_exit_debit_points: float
    pnl_points: float
    lot_size: int
    lots: int
    pnl_rupees: float
    capital_before: float
    capital_after: float

    def to_dict(self) -> dict[str, object]:
        return asdict(self)


@dataclass(frozen=True)
class Skip:
    mode: str
    expiry: str
    archive: str
    reason: str
    detail: str = ""

    def to_dict(self) -> dict[str, object]:
        return asdict(self)


def parse_bar(row: dict[str, str]) -> Bar:
    return Bar(
        timestamp=datetime.strptime(row["Timestamp"], TIMESTAMP_FORMAT),
        open=float(row["Open"]),
        high=float(row["High"]),
        low=float(row["Low"]),
        close=float(row["Close"]),
        volume=int(float(row["Volume"])),
        oi=int(float(row["OI"])),
        ticker=row["Ticker"],
    )


def read_option_bars(archive: zipfile.ZipFile, member: str) -> dict[datetime, Bar]:
    with archive.open(member) as raw:
        reader = csv.DictReader(io.TextIOWrapper(raw, encoding="utf-8-sig", newline=""))
        bars: dict[datetime, Bar] = {}
        for row in reader:
            bar = parse_bar(row)
            if bar.timestamp in bars:
                raise ValueError(f"Duplicate timestamp in {member}: {bar.timestamp}")
            bars[bar.timestamp] = bar
    return bars


def load_spot_bars(path: Path) -> dict[datetime, Bar]:
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        bars: dict[datetime, Bar] = {}
        for row in reader:
            bar = parse_bar(row)
            if bar.timestamp in bars:
                raise ValueError(f"Duplicate consolidated spot timestamp: {bar.timestamp}")
            bars[bar.timestamp] = bar
    return bars


def load_complete_dates(path: Path) -> list[date]:
    with path.open(encoding="utf-8", newline="") as handle:
        rows = csv.DictReader(handle)
        return [
            date.fromisoformat(row["date"])
            for row in rows
            if row["regular_session_complete"].lower() == "true"
        ]


def build_daily_closes(
    spot_bars: dict[datetime, Bar], complete_dates: Iterable[date]
) -> dict[date, float]:
    result: dict[date, float] = {}
    for trading_date in complete_dates:
        timestamp = datetime.combine(trading_date, time(15, 30))
        bar = spot_bars.get(timestamp)
        if bar is None:
            candidates = [
                item
                for item in spot_bars.values()
                if item.timestamp.date() == trading_date
                and time(15, 15) <= item.timestamp.time() <= time(15, 30)
            ]
            if not candidates:
                continue
            bar = max(candidates, key=lambda item: item.timestamp)
        result[trading_date] = bar.close
    return result


def calculate_regime_features(
    prior_closes: list[float],
    spot_at_signal: float,
    atm_straddle_price: float,
    holding_sessions: int = 5,
    lookback: int = 20,
) -> RegimeFeatures:
    if len(prior_closes) < max(lookback + 1, holding_sessions + 1):
        raise ValueError("Insufficient prior closes")
    if spot_at_signal <= 0 or atm_straddle_price <= 0:
        raise ValueError("Spot and straddle price must be positive")
    log_returns = [
        math.log(current / previous)
        for previous, current in zip(prior_closes[:-1], prior_closes[1:])
    ]
    recent_returns = log_returns[-lookback:]
    sigma = statistics.stdev(recent_returns)
    if sigma <= 0:
        raise ValueError("Realized volatility is zero")
    expected_abs = sigma * math.sqrt(holding_sessions) * math.sqrt(2 / math.pi)
    implied_move_pct = atm_straddle_price / spot_at_signal
    trend_return = math.log(prior_closes[-1] / prior_closes[-(holding_sessions + 1)])
    trend_z = abs(trend_return) / (sigma * math.sqrt(holding_sessions))
    return RegimeFeatures(
        daily_sigma=sigma,
        expected_abs_realized_move_pct=expected_abs,
        implied_move_pct=implied_move_pct,
        vrp_ratio=implied_move_pct / expected_abs,
        five_day_trend_log_return=trend_return,
        trend_z=trend_z,
    )


def lot_size_for_expiry(expiry: date) -> int:
    """Historical NIFTY market lots for expiries in the supplied 2024-25 data."""
    if expiry <= date(2024, 4, 25):
        return 50
    if expiry <= date(2024, 12, 26):
        return 25
    if expiry == date(2025, 1, 30):
        return 25  # final monthly expiry retaining the pre-revision lot
    return 75


def adverse_sell_fill(market_price: float, friction: float) -> float:
    if market_price <= 0:
        raise ValueError("Market price must be positive")
    return market_price * (1 - friction)


def adverse_buy_fill(market_price: float, friction: float) -> float:
    if market_price < 0:
        raise ValueError("Market price cannot be negative")
    return market_price * (1 + friction)


def position_size(
    capital: float,
    spot: float,
    gross_credit_points: float,
    lot_size: int,
    config: StrategyConfig,
) -> int:
    """Integer lots capped independently by stop-risk and margin proxies."""
    stop_loss_points = (config.stop_multiple - 1) * gross_credit_points
    # Ten percent buffer covers friction and gaps through the stop level.
    risk_per_lot = stop_loss_points * lot_size * 1.10
    margin_per_lot = config.margin_proxy_fraction * spot * lot_size
    risk_lots = math.floor(capital * config.risk_fraction / risk_per_lot)
    margin_lots = math.floor(
        capital * config.margin_allocation_fraction / margin_per_lot
    )
    return max(0, min(risk_lots, margin_lots))


def member_name(strike: int, right: str, expiry: date) -> str:
    return f"{strike}{right}_{expiry:%Y%m%d}.csv"


def paired_strikes(archive: zipfile.ZipFile, expiry: date) -> list[int]:
    calls: set[int] = set()
    puts: set[int] = set()
    for name in archive.namelist():
        match = OPTION_MEMBER_RE.match(name)
        if not match or match.group("expiry") != f"{expiry:%Y%m%d}":
            continue
        strike = int(match.group("strike"))
        (calls if match.group("right") == "CE" else puts).add(strike)
    return sorted(calls & puts)


def find_common_fill_timestamp(
    call_bars: dict[datetime, Bar],
    put_bars: dict[datetime, Bar],
    planned: datetime,
    max_delay_minutes: int,
) -> datetime | None:
    deadline = planned + timedelta(minutes=max_delay_minutes)
    common = call_bars.keys() & put_bars.keys()
    candidates = [timestamp for timestamp in common if planned <= timestamp <= deadline]
    return min(candidates) if candidates else None


def _valid_signal_bar(bar: Bar | None) -> bool:
    return bool(bar and bar.close > 0 and bar.oi > 0)


def _candidate_contract(
    archive: zipfile.ZipFile,
    expiry: date,
    strikes: Iterable[int],
    right: str,
    signal_timestamp: datetime,
    cache: dict[str, dict[datetime, Bar]],
) -> tuple[int, dict[datetime, Bar]] | None:
    for strike in strikes:
        name = member_name(strike, right, expiry)
        bars = cache.setdefault(name, read_option_bars(archive, name))
        if _valid_signal_bar(bars.get(signal_timestamp)):
            return strike, bars
    return None


def _next_common_timestamp(
    call_bars: dict[datetime, Bar],
    put_bars: dict[datetime, Bar],
    after: datetime,
    no_later_than: datetime,
) -> datetime | None:
    candidates = [
        timestamp
        for timestamp in call_bars.keys() & put_bars.keys()
        if after < timestamp <= no_later_than
    ]
    return min(candidates) if candidates else None


def evaluate_expiry(
    archive_path: Path,
    expiry: date,
    spot_bars: dict[datetime, Bar],
    complete_dates: list[date],
    daily_closes: dict[date, float],
    capital: float,
    config: StrategyConfig,
    entry_date_override: date | None = None,
    entry_not_before: datetime | None = None,
) -> Trade | Skip:
    config.validate()
    mode = "filtered" if config.use_regime_filter else "benchmark"
    if expiry > DATA_CUTOFF:
        return Skip(mode, str(expiry), archive_path.name, "post_cutoff")
    complete_before_or_on = [item for item in complete_dates if item <= expiry]
    if expiry not in complete_before_or_on:
        return Skip(mode, str(expiry), archive_path.name, "incomplete_expiry_session")
    expiry_index = complete_before_or_on.index(expiry)
    if entry_date_override is None:
        if expiry_index < config.fallback_holding_sessions:
            return Skip(mode, str(expiry), archive_path.name, "insufficient_calendar_history")
        entry_date = complete_before_or_on[
            expiry_index - config.fallback_holding_sessions
        ]
    else:
        entry_date = entry_date_override
        if entry_date not in complete_before_or_on or entry_date >= expiry:
            return Skip(
                mode,
                str(expiry),
                archive_path.name,
                "invalid_previous_expiry_entry",
                str(entry_date),
            )
    entry_index = complete_before_or_on.index(entry_date)
    holding_sessions = expiry_index - entry_index
    if holding_sessions <= 0:
        return Skip(mode, str(expiry), archive_path.name, "nonpositive_holding_period")
    prior_dates = [item for item in complete_dates if item < entry_date and item in daily_closes]
    if len(prior_dates) < config.realized_lookback + 1:
        return Skip(mode, str(expiry), archive_path.name, "insufficient_feature_history")
    feature_dates = prior_dates[-(config.realized_lookback + 1) :]
    prior_closes = [daily_closes[item] for item in feature_dates]
    signal_timestamp = datetime.combine(entry_date, config.signal_time)
    signal_spot_bar = spot_bars.get(signal_timestamp)
    if signal_spot_bar is None or signal_spot_bar.close <= 0:
        return Skip(mode, str(expiry), archive_path.name, "missing_signal_spot")

    cache: dict[str, dict[datetime, Bar]] = {}
    with zipfile.ZipFile(archive_path) as archive:
        strikes = paired_strikes(archive, expiry)
        if not strikes:
            return Skip(mode, str(expiry), archive_path.name, "no_paired_strikes")

        nearest = sorted(strikes, key=lambda strike: (abs(strike - signal_spot_bar.close), strike))
        atm: tuple[int, dict[datetime, Bar], dict[datetime, Bar]] | None = None
        for strike in nearest:
            call_name = member_name(strike, "CE", expiry)
            put_name = member_name(strike, "PE", expiry)
            call_bars = cache.setdefault(call_name, read_option_bars(archive, call_name))
            put_bars = cache.setdefault(put_name, read_option_bars(archive, put_name))
            if _valid_signal_bar(call_bars.get(signal_timestamp)) and _valid_signal_bar(
                put_bars.get(signal_timestamp)
            ):
                atm = strike, call_bars, put_bars
                break
        if atm is None:
            return Skip(mode, str(expiry), archive_path.name, "no_contemporaneous_atm_pair")
        atm_strike, atm_call_bars, atm_put_bars = atm
        atm_call = atm_call_bars[signal_timestamp]
        atm_put = atm_put_bars[signal_timestamp]
        implied_move_points = atm_call.close + atm_put.close
        features = calculate_regime_features(
            prior_closes,
            signal_spot_bar.close,
            implied_move_points,
            holding_sessions,
            config.realized_lookback,
        )
        if (
            config.use_regime_filter
            and config.vrp_filter_direction == "above"
            and features.vrp_ratio < config.minimum_vrp_ratio
        ):
            return Skip(
                mode,
                str(expiry),
                archive_path.name,
                "vrp_filter",
                f"{features.vrp_ratio:.6f} < {config.minimum_vrp_ratio:.6f}",
            )
        if (
            config.use_regime_filter
            and config.vrp_filter_direction == "below"
            and features.vrp_ratio > config.maximum_vrp_ratio
        ):
            return Skip(
                mode,
                str(expiry),
                archive_path.name,
                "vrp_filter",
                f"{features.vrp_ratio:.6f} > {config.maximum_vrp_ratio:.6f}",
            )
        if (
            config.use_regime_filter
            and config.use_trend_filter
            and features.trend_z > config.maximum_trend_z
        ):
            return Skip(
                mode,
                str(expiry),
                archive_path.name,
                "trend_filter",
                f"{features.trend_z:.6f} > {config.maximum_trend_z:.6f}",
            )

        width = config.strike_width_multiplier * implied_move_points
        call_target = signal_spot_bar.close + width
        put_target = signal_spot_bar.close - width
        call_candidates = [strike for strike in strikes if strike >= call_target]
        put_candidates = [strike for strike in reversed(strikes) if strike <= put_target]
        call_contract = _candidate_contract(
            archive, expiry, call_candidates, "CE", signal_timestamp, cache
        )
        put_contract = _candidate_contract(
            archive, expiry, put_candidates, "PE", signal_timestamp, cache
        )
        if call_contract is None or put_contract is None:
            return Skip(mode, str(expiry), archive_path.name, "no_liquid_outward_strangle")
        call_strike, call_bars = call_contract
        put_strike, put_bars = put_contract

        base_planned_entry = datetime.combine(entry_date, config.planned_fill_time)
        planned_entry = max(
            base_planned_entry,
            entry_not_before or base_planned_entry,
        )
        entry_deadline = base_planned_entry + timedelta(
            minutes=config.max_fill_delay_minutes
        )
        remaining_delay = int((entry_deadline - planned_entry).total_seconds() // 60)
        if remaining_delay < 0:
            return Skip(mode, str(expiry), archive_path.name, "prior_position_not_closed")
        entry_timestamp = find_common_fill_timestamp(
            call_bars, put_bars, planned_entry, remaining_delay
        )
        if entry_timestamp is None:
            return Skip(mode, str(expiry), archive_path.name, "missing_entry_fill")
        call_entry_market = call_bars[entry_timestamp].open
        put_entry_market = put_bars[entry_timestamp].open
        if min(call_entry_market, put_entry_market) <= 0:
            return Skip(mode, str(expiry), archive_path.name, "nonpositive_entry_price")
        gross_entry_mark = call_entry_market + put_entry_market
        call_entry_fill = adverse_sell_fill(call_entry_market, config.execution_friction)
        put_entry_fill = adverse_sell_fill(put_entry_market, config.execution_friction)
        net_entry_credit = call_entry_fill + put_entry_fill
        lot_size = lot_size_for_expiry(expiry)
        lots = position_size(
            capital, signal_spot_bar.close, gross_entry_mark, lot_size, config
        )
        if lots < 1:
            return Skip(mode, str(expiry), archive_path.name, "position_size_zero")

        expiry_planned_exit = datetime.combine(expiry, config.planned_fill_time)
        expiry_fill_deadline = expiry_planned_exit + timedelta(
            minutes=config.max_fill_delay_minutes
        )
        common_timestamps = sorted(call_bars.keys() & put_bars.keys())
        trigger_timestamp: datetime | None = None
        exit_timestamp: datetime | None = None
        exit_reason = "time_exit"
        for timestamp in common_timestamps:
            if not entry_timestamp <= timestamp < expiry_planned_exit:
                continue
            combined_close = call_bars[timestamp].close + put_bars[timestamp].close
            reason: str | None = None
            if combined_close <= config.profit_target_fraction * gross_entry_mark:
                reason = "profit_target"
            elif combined_close >= config.stop_multiple * gross_entry_mark:
                reason = "stop_loss"
            if reason:
                next_timestamp = _next_common_timestamp(
                    call_bars,
                    put_bars,
                    timestamp,
                    min(
                        timestamp + timedelta(minutes=config.max_fill_delay_minutes),
                        expiry_fill_deadline,
                    ),
                )
                if next_timestamp is not None:
                    trigger_timestamp = timestamp
                    exit_timestamp = next_timestamp
                    exit_reason = reason
                    break

        if exit_timestamp is None:
            exit_timestamp = find_common_fill_timestamp(
                call_bars,
                put_bars,
                expiry_planned_exit,
                config.max_fill_delay_minutes,
            )
        if exit_timestamp is None:
            return Skip(mode, str(expiry), archive_path.name, "missing_exit_fill")

        call_exit_market = call_bars[exit_timestamp].open
        put_exit_market = put_bars[exit_timestamp].open
        if min(call_exit_market, put_exit_market) < 0:
            return Skip(mode, str(expiry), archive_path.name, "negative_exit_price")
        call_exit_fill = adverse_buy_fill(call_exit_market, config.execution_friction)
        put_exit_fill = adverse_buy_fill(put_exit_market, config.execution_friction)
        net_exit_debit = call_exit_fill + put_exit_fill
        pnl_points = net_entry_credit - net_exit_debit
        pnl_rupees = pnl_points * lot_size * lots
        capital_after = capital + pnl_rupees

        if not signal_timestamp < entry_timestamp <= exit_timestamp:
            raise AssertionError("Signal/fill/exit chronology is invalid")
        if not put_strike < signal_spot_bar.close < call_strike:
            raise AssertionError("Selected contracts do not form an OTM strangle")

        return Trade(
            mode=mode,
            expiry=str(expiry),
            archive=archive_path.name,
            entry_date=str(entry_date),
            holding_sessions=holding_sessions,
            feature_end_date=str(feature_dates[-1]),
            signal_timestamp=signal_timestamp.isoformat(sep=" "),
            entry_timestamp=entry_timestamp.isoformat(sep=" "),
            trigger_timestamp=trigger_timestamp.isoformat(sep=" ")
            if trigger_timestamp
            else "",
            exit_timestamp=exit_timestamp.isoformat(sep=" "),
            exit_reason=exit_reason,
            spot_at_signal=signal_spot_bar.close,
            atm_strike=atm_strike,
            atm_call_close=atm_call.close,
            atm_put_close=atm_put.close,
            implied_move_points=implied_move_points,
            implied_move_pct=features.implied_move_pct,
            expected_abs_realized_move_pct=features.expected_abs_realized_move_pct,
            vrp_ratio=features.vrp_ratio,
            trend_z=features.trend_z,
            put_strike=put_strike,
            call_strike=call_strike,
            put_entry_market=put_entry_market,
            call_entry_market=call_entry_market,
            gross_entry_mark=gross_entry_mark,
            put_entry_fill=put_entry_fill,
            call_entry_fill=call_entry_fill,
            net_entry_credit_points=net_entry_credit,
            put_exit_market=put_exit_market,
            call_exit_market=call_exit_market,
            put_exit_fill=put_exit_fill,
            call_exit_fill=call_exit_fill,
            net_exit_debit_points=net_exit_debit,
            pnl_points=pnl_points,
            lot_size=lot_size,
            lots=lots,
            pnl_rupees=pnl_rupees,
            capital_before=capital,
            capital_after=capital_after,
        )


def archive_expiry(path: Path) -> date:
    return datetime.strptime(path.stem, "%Y%m%d").date()

## Data loading and cleaning

Each ZIP repeats part of the NIFTY spot history. The pipeline reads only
eligible archives, verifies repeated OHLC values, deduplicates by
timestamp, and keeps a source-archive field for traceability. Raw files
are never modified or extracted en masse.

In [3]:
eligible_archives = sorted(
    path for path in DATA_ROOT.glob("*.zip")
    if path.stem.isdigit()
    and len(path.stem) == 8
    and path.stem <= "20251130"
)
excluded_archives = sorted(
    path for path in DATA_ROOT.glob("*.zip")
    if path.stem.isdigit() and path.stem > "20251130"
)

spot_rows = {}
source_archive = {}
duplicate_rows = 0
conflicting_duplicates = 0
option_member_count = 0
schema_counts = Counter()
for archive_path in eligible_archives:
    with zipfile.ZipFile(archive_path) as archive:
        option_member_count += sum(
            bool(OPTION_MEMBER_RE.match(name)) for name in archive.namelist()
        )
        with archive.open("nifty_spot.csv") as raw:
            reader = csv.DictReader(
                io.TextIOWrapper(raw, encoding="utf-8-sig", newline="")
            )
            schema_counts[tuple(reader.fieldnames or [])] += 1
            for row in reader:
                timestamp = datetime.strptime(row["Timestamp"], TIMESTAMP_FORMAT)
                prior = spot_rows.get(timestamp)
                if prior is not None:
                    duplicate_rows += 1
                    if any(
                        prior[key] != row[key]
                        for key in ("Open", "High", "Low", "Close", "Ticker")
                    ):
                        conflicting_duplicates += 1
                else:
                    spot_rows[timestamp] = row
                    source_archive[timestamp] = archive_path.name

if conflicting_duplicates:
    raise AssertionError("Conflicting duplicated spot observations found")
if len(schema_counts) != 1:
    raise AssertionError(f"Unexpected spot schemas: {schema_counts}")

spot_cache = OUTPUT_ROOT / "nifty_spot_1min.csv.gz"
fields = ["Date", "Timestamp", "Open", "High", "Low", "Close", "Volume", "OI", "Ticker"]
with gzip.open(spot_cache, "wt", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields + ["SourceArchive"])
    writer.writeheader()
    for timestamp in sorted(spot_rows):
        row = {key: spot_rows[timestamp][key] for key in fields}
        row["SourceArchive"] = source_archive[timestamp]
        writer.writerow(row)

audit = {
    "eligible_archives": len(eligible_archives),
    "excluded_post_cutoff_archives": len(excluded_archives),
    "option_csv_members": option_member_count,
    "spot_rows_scanned": len(spot_rows) + duplicate_rows,
    "unique_spot_rows": len(spot_rows),
    "duplicate_spot_rows_removed": duplicate_rows,
    "conflicting_duplicates": conflicting_duplicates,
    "spot_start": min(spot_rows).isoformat(sep=" "),
    "spot_end": max(spot_rows).isoformat(sep=" "),
}
print(json.dumps(audit, indent=2))

{
  "eligible_archives": 101,
  "excluded_post_cutoff_archives": 24,
  "option_csv_members": 19520,
  "spot_rows_scanned": 1529995,
  "unique_spot_rows": 177472,
  "duplicate_spot_rows_removed": 1352523,
  "conflicting_duplicates": 0,
  "spot_start": "2024-01-01 09:07:00",
  "spot_end": "2025-11-25 15:31:00"
}


Six short or special market sessions exist in the source history. A date
is treated as a complete regular session only when it contains at least
370 observations between 09:15 and 15:30. Entries and lagged daily closes
use only complete sessions.

In [4]:
spot_bars = load_spot_bars(spot_cache)
regular_counts = Counter(
    timestamp.date()
    for timestamp in spot_bars
    if time(9, 15) <= timestamp.time() <= time(15, 30)
)
complete_dates = sorted(
    trading_date for trading_date, count in regular_counts.items() if count >= 370
)
partial_dates = sorted(
    (str(trading_date), count)
    for trading_date, count in regular_counts.items()
    if count < 370
)
daily_closes = build_daily_closes(spot_bars, complete_dates)
print(f"Complete sessions: {len(complete_dates)}")
print(f"Partial/special sessions: {partial_dates}")

Complete sessions: 467
Partial/special sessions: [('2024-03-02', 109), ('2024-05-18', 111), ('2025-03-05', 342), ('2025-10-21', 62), ('2025-11-10', 366)]


## Backtest

The aftershock threshold is the neutral equality boundary `priced move /
realized comparator <= 1.0`. The benchmark uses identical strikes,
sizing, costs, and exits but trades every expiry for which the feature and
quote data are available.

In [5]:
strategy_config = StrategyConfig(
    vrp_filter_direction="below",
    maximum_vrp_ratio=1.0,
    use_trend_filter=False,
)
benchmark_config = replace(strategy_config, use_regime_filter=False)

def run_variant(config):
    capital = config.initial_capital
    last_exit = None
    trades, skips = [], []
    for index, archive_path in enumerate(eligible_archives):
        expiry = archive_expiry(archive_path)
        previous_expiry = (
            archive_expiry(eligible_archives[index - 1]) if index else None
        )
        result = evaluate_expiry(
            archive_path, expiry, spot_bars, complete_dates, daily_closes,
            capital, config, previous_expiry, last_exit
        )
        if isinstance(result, Trade):
            trades.append(result)
            capital = result.capital_after
            last_exit = datetime.fromisoformat(result.exit_timestamp)
        else:
            skips.append(result)
    return trades, skips

strategy_trades, strategy_skips = run_variant(strategy_config)
benchmark_trades, benchmark_skips = run_variant(benchmark_config)

def export_records(path, records):
    rows = [record.to_dict() for record in records]
    with path.open("w", encoding="utf-8", newline="") as handle:
        if rows:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader(); writer.writerows(rows)

export_records(OUTPUT_ROOT / "strategy_trades.csv", strategy_trades)
export_records(OUTPUT_ROOT / "strategy_skips.csv", strategy_skips)
export_records(OUTPUT_ROOT / "benchmark_trades.csv", benchmark_trades)
print(f"Strategy trades: {len(strategy_trades)}; skips: {len(strategy_skips)}")
print(f"Benchmark trades: {len(benchmark_trades)}; skips: {len(benchmark_skips)}")

Strategy trades: 26; skips: 75
Benchmark trades: 95; skips: 6


## Performance and robustness

Returns are measured once per candidate expiry; non-trading expiries
receive zero return. Annualized volatility and Sharpe therefore use 52
expiry observations per year and a zero risk-free rate. Drawdown is based
on realized weekly equity and does not include intratrade mark-to-market,
which is disclosed as a limitation.

In [6]:
def candidate_rows(trades):
    return [trade.to_dict() for trade in trades]

all_candidates = candidate_rows(benchmark_trades)

def integer_lots_from_row(capital, row):
    gross = float(row["gross_entry_mark"]); spot = float(row["spot_at_signal"])
    lot = int(row["lot_size"])
    risk_lots = math.floor(capital * 0.01 / (gross * lot * 1.10))
    margin_lots = math.floor(capital * 0.25 / (0.10 * spot * lot))
    return max(0, min(risk_lots, margin_lots))

def analyze_threshold(threshold=1.0, friction=0.005):
    capital = 2_000_000.0; peak = capital; max_dd = 0.0
    weekly_returns, selected, equity = [], [], []
    for row in all_candidates:
        before = capital; pnl = 0.0; lots = 0
        if float(row["vrp_ratio"]) <= threshold:
            lots = integer_lots_from_row(capital, row)
            if lots:
                entry = float(row["gross_entry_mark"])
                exit_mark = float(row["put_exit_market"]) + float(row["call_exit_market"])
                points = entry * (1 - friction) - exit_mark * (1 + friction)
                pnl = points * int(row["lot_size"]) * lots
                capital += pnl
                selected.append({**row, "recomputed_pnl": pnl})
        weekly_returns.append(pnl / before)
        peak = max(peak, capital); dd = capital / peak - 1; max_dd = min(max_dd, dd)
        equity.append((row["expiry"], capital, dd))
    pnl_values = [row["recomputed_pnl"] for row in selected]
    wins = [value for value in pnl_values if value > 0]
    losses = [value for value in pnl_values if value < 0]
    elapsed = max(1, (
        datetime.fromisoformat(all_candidates[-1]["exit_timestamp"])
        - datetime.fromisoformat(all_candidates[0]["entry_timestamp"])
    ).days)
    vol = statistics.stdev(weekly_returns) * math.sqrt(52)
    return {
        "threshold": threshold, "friction": friction,
        "trades": len(selected), "total_pnl": capital - 2_000_000,
        "total_return": capital / 2_000_000 - 1,
        "annualized_return": (capital / 2_000_000) ** (365 / elapsed) - 1,
        "annualized_volatility": vol,
        "sharpe_zero_rf": statistics.mean(weekly_returns) / statistics.stdev(weekly_returns) * math.sqrt(52),
        "maximum_realized_drawdown": max_dd,
        "win_rate": len(wins) / len(selected),
        "average_payoff": statistics.mean(pnl_values),
        "payoff_ratio": statistics.mean(wins) / abs(statistics.mean(losses)) if wins and losses else None,
        "profit_factor": sum(wins) / abs(sum(losses)) if wins and losses else None,
        "equity": equity,
        "selected": selected,
    }

base = analyze_threshold()
threshold_sensitivity = [analyze_threshold(value) for value in (0.8, 0.9, 1.0, 1.1, 1.2)]
cost_sensitivity = [analyze_threshold(1.0, value) for value in (0.0, 0.0025, 0.005, 0.01)]
printable_base = {key: value for key, value in base.items() if key not in {"equity", "selected"}}
print(json.dumps(printable_base, indent=2))
print("Threshold sensitivity (threshold, trades, return):")
print([(row["threshold"], row["trades"], round(row["total_return"], 4)) for row in threshold_sensitivity])
print("Cost sensitivity (friction, return):")
print([(row["friction"], round(row["total_return"], 4)) for row in cost_sensitivity])

{
  "threshold": 1.0,
  "friction": 0.005,
  "trades": 26,
  "total_pnl": 47142.43749999977,
  "total_return": 0.0235712187499999,
  "annualized_return": 0.012987503884649865,
  "annualized_volatility": 0.018325121715362797,
  "sharpe_zero_rf": 0.7050731164623617,
  "maximum_realized_drawdown": -0.024503603347737934,
  "win_rate": 0.7692307692307693,
  "average_payoff": 1813.1706730769242,
  "payoff_ratio": 0.4527855528321954,
  "profit_factor": 1.5092851761073178
}
Threshold sensitivity (threshold, trades, return):
[(0.8, 10, 0.0335), (0.9, 20, 0.0158), (1.0, 26, 0.0236), (1.1, 44, 0.014), (1.2, 54, 0.0191)]
Cost sensitivity (friction, return):
[(0.0, 0.0253), (0.0025, 0.0244), (0.005, 0.0236), (0.01, 0.0218)]


In [7]:
def line_svg(series, path, title, y_label, percent=False):
    width, height, pad = 900, 360, 60
    values = [value for _, value in series]
    lo, hi = min(values), max(values)
    if math.isclose(lo, hi): hi = lo + 1
    def xy(index, value):
        x = pad + index * (width - 2 * pad) / max(1, len(series) - 1)
        y = height - pad - (value - lo) * (height - 2 * pad) / (hi - lo)
        return x, y
    points = " ".join(f"{xy(i,v)[0]:.1f},{xy(i,v)[1]:.1f}" for i,(_,v) in enumerate(series))
    label_lo = f"{lo:.1%}" if percent else f"{lo:,.0f}"
    label_hi = f"{hi:.1%}" if percent else f"{hi:,.0f}"
    svg = f'''<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">
    <rect width="100%" height="100%" fill="white"/><text x="{pad}" y="30" font-size="20" font-family="Arial">{title}</text>
    <line x1="{pad}" y1="{pad}" x2="{pad}" y2="{height-pad}" stroke="#666"/><line x1="{pad}" y1="{height-pad}" x2="{width-pad}" y2="{height-pad}" stroke="#666"/>
    <polyline fill="none" stroke="#9b6a20" stroke-width="3" points="{points}"/>
    <text x="5" y="{pad+5}" font-size="12" font-family="Arial">{label_hi}</text><text x="5" y="{height-pad+5}" font-size="12" font-family="Arial">{label_lo}</text>
    <text x="{width/2-30}" y="{height-15}" font-size="12" font-family="Arial">Expiry sequence</text><text x="5" y="45" font-size="12" font-family="Arial">{y_label}</text></svg>'''
    path.write_text(svg, encoding="utf-8")

line_svg([(expiry, equity) for expiry, equity, _ in base["equity"]], OUTPUT_ROOT / "equity_curve.svg", "Aftershock strategy realized equity", "INR")
line_svg([(expiry, dd) for expiry, _, dd in base["equity"]], OUTPUT_ROOT / "drawdown.svg", "Aftershock strategy realized drawdown", "Drawdown", percent=True)
print("Charts written to", OUTPUT_ROOT)

Charts written to deliverables/notebook_outputs


## Results

The base strategy executes 26 trades. It earns approximately **2.36% in
total** and **1.30% annualized**, with **1.83% annualized weekly
volatility**, a **0.71 zero-rate Sharpe ratio**, **2.45% maximum realized
drawdown**, and **76.9% win rate**. Average winners are smaller than
average losses, as expected for a short-volatility strategy; the profit
factor is approximately **1.51**.

The threshold check remains profitable from 0.8 through 1.2. The cost
check remains profitable when friction doubles to 1% on every
transaction.

![Realized equity curve](deliverables/notebook_outputs/equity_curve.svg)

![Realized drawdown](deliverables/notebook_outputs/drawdown.svg)

## Discussion and limitations

The evidence supports a conditional mean-reversion interpretation, not a
universal option-selling premium. The original high-priced-move filter
failed and is retained in the research record rather than hidden.

Important limitations are the short January 2024–November 2025 sample,
only 26 strategy trades, last-traded prices rather than bid/ask quotes,
simplified proportional execution costs, a margin proxy rather than
exchange SPAN files, and realized rather than intratrade mark-to-market
drawdown. The 2025 H2 stability segment contains only two trades. Results
should therefore be treated as preliminary evidence, not a production
trading recommendation.

A Round 2 extension should add bid/ask-aware fills, exact SPAN margin,
intratrade marked equity, event labels, an independently reserved future
sample, and comparison with Black–Scholes IV/delta-based strike selection.